# Tuning Neural Networks with Normalization - Lab 

## Introduction

In this lab you'll build a neural network to perform a regression task.

It is worth noting that getting regression to work with neural networks can be comparatively difficult because the output is unbounded ($\hat y$ can technically range from $-\infty$ to $+\infty$), and the models are especially prone to exploding gradients. This issue makes a regression exercise the perfect learning case for tinkering with normalization and optimization strategies to ensure proper convergence!

## Objectives

In this lab you will: 

- Fit a neural network to normalized data 
- Implement and observe the impact of various initialization techniques 
- Implement and observe the impact of various optimization techniques 

## Load the data 

First, run the following cell to import all the neccessary libraries and classes you will need in this lab. 

In [1]:
# Necessary libraries and classes
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras import initializers
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from keras import optimizers
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In this lab, you'll be working with the housing prices data you saw in an earlier section. However, we did a lot of preprocessing for you so you can focus on normalizing numeric features and building neural network models! The following preprocessing steps were taken (all the code can be found in the `data_preprocessing.ipynb` notebook in this repository): 

- The data was split into the training, validate, and test sets 
- All the missing values in numeric columns were replaced by the median of those columns 
- All the missing values in catetgorical columns were replaced with the word 'missing' 
- All the categorical columns were one-hot encoded 

Run the following cells to import the train, validate, and test sets:  

In [2]:
# Load all numeric features
X_train_numeric = pd.read_csv('data/X_train_numeric.csv')
X_val_numeric = pd.read_csv('data/X_val_numeric.csv')
X_test_numeric = pd.read_csv('data/X_test_numeric.csv')

# Load all categorical features
X_train_cat = pd.read_csv('data/X_train_cat.csv')
X_val_cat = pd.read_csv('data/X_val_cat.csv')
X_test_cat = pd.read_csv('data/X_test_cat.csv')

# Load all targets
y_train = pd.read_csv('data/y_train.csv')
y_val = pd.read_csv('data/y_val.csv')
y_test = pd.read_csv('data/y_test.csv')

In [3]:
# Combine all features
X_train = pd.concat([X_train_numeric, X_train_cat], axis=1)
X_val = pd.concat([X_val_numeric, X_val_cat], axis=1)
X_test = pd.concat([X_test_numeric, X_test_cat], axis=1)

# Number of features
n_features = X_train.shape[1]

As a refresher, preview the training data: 

In [4]:
# Preview the data
X_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,80.0,69.0,21453.0,6.0,5.0,1969.0,1969.0,0.0,938.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,60.0,79.0,12420.0,7.0,5.0,2001.0,2001.0,0.0,666.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,20.0,75.0,9742.0,8.0,5.0,2002.0,2002.0,281.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,120.0,39.0,5389.0,8.0,5.0,1995.0,1996.0,0.0,1180.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,60.0,85.0,11003.0,10.0,5.0,2008.0,2008.0,160.0,765.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## Build a Baseline Model

Building a naive baseline model to compare performance against is a helpful reference point. From there, you can then observe the impact of various tunning procedures which will iteratively improve your model. So, let's do just that! 

In the cell below: 

- Add an input layer with `n_features` units 
- Add two hidden layers, one with 100 and the other with 50 units (make sure you use the `'relu'` activation function) 
- Add an output layer with 1 unit and `'linear'` activation 
- Compile and fit the model 

In [5]:
np.random.seed(123)
baseline_model = Sequential()

# Hidden layer with 100 units
baseline_model.add(layers.Dense(100,activation='relu',input_shape=(n_features,)))

# Hidden layer with 50 units
baseline_model.add(layers.Dense(50,activation='relu'))

# Output layer
baseline_model.add(layers.Dense(1,activation='linear'))

# Compile the model
baseline_model.compile(optimizer='SGD', 
                       loss='mse', 
                       metrics=['mse'])

# Train the model
baseline_model.fit(X_train, 
                   y_train, 
                   batch_size=32, 
                   epochs=150, 
                   validation_data=(X_val, y_val))

Epoch 1/150
33/33 [==============================] - 0s 4ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 9/150
33/33 [=============================

33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 71/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 72/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 73/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 74/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 75/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 76/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 77/150
33/33 [==============================] -

33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 138/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 139/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 140/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 141/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 142/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 143/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 144/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 145/150
33/33 [=========================

> _**Notice this extremely problematic behavior: all the values for training and validation loss are "nan". This indicates that the algorithm did not converge. The first solution to this is to normalize the input. From there, if convergence is not achieved, normalizing the output may also be required.**_ 

## Normalize the Input Data 

It's now time to normalize the input data. In the cell below: 

- Assign the column names of all numeric columns to `numeric_columns` 
- Instantiate a `StandardScaler` 
- Fit and transform `X_train_numeric`. Make sure you convert the result into a DataFrame (use `numeric_columns` as the column names) 
- Transform validate and test sets (`X_val_numeric` and `X_test_numeric`), and convert these results into DataFrames as well 
- Use the provided to combine the scaled numerical and categorical features 

In [7]:
# Numeric column names
numeric_columns = X_train_numeric.columns 

# Instantiate StandardScaler
ss_X = StandardScaler()

# Fit and transform train data
X_train_scaled = pd.DataFrame(ss_X.fit_transform(X_train_numeric), columns=numeric_columns)

# Transform validate and test data
X_val_scaled = pd.DataFrame(ss_X.transform(X_val_numeric), columns=numeric_columns)
X_test_scaled = pd.DataFrame(ss_X.transform(X_test_numeric), columns=numeric_columns)

# Combine the scaled numerical features and categorical features
X_train = pd.concat([X_train_scaled, X_train_cat], axis=1)
X_val = pd.concat([X_val_scaled, X_val_cat], axis=1)
X_test = pd.concat([X_test_scaled, X_test_cat], axis=1)


Now run the following cell to compile a neural network model (with the same architecture as before): 

In [8]:
# Model with all normalized inputs
np.random.seed(123)
normalized_input_model = Sequential()
normalized_input_model.add(layers.Dense(100, activation='relu', input_shape=(n_features,)))
normalized_input_model.add(layers.Dense(50, activation='relu'))
normalized_input_model.add(layers.Dense(1, activation='linear'))

# Compile the model
normalized_input_model.compile(optimizer='SGD', 
                               loss='mse', 
                               metrics=['mse'])

In the cell below: 
- Train the `normalized_input_model` on normalized input (`X_train`) and output (`y_train`) 
- Set a batch size of 32 and train for 150 epochs 
- Specify the `validation_data` argument as `(X_val, y_val)` 

In [9]:
# Train the model 
normalized_input_model.fit(X_train,y_train,batch_size=32,epochs=150,validation_data=(X_val,y_val))

Epoch 1/150
33/33 [==============================] - 0s 4ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 9/150
33/33 [=============================

33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 71/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 72/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 73/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 74/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 75/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 76/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 77/150
33/33 [==============================] -

33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 138/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 139/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 140/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 141/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 142/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 143/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 144/150
33/33 [==============================] - 0s 1ms/step - loss: nan - mse: nan - val_loss: nan - val_mse: nan
Epoch 145/150
33/33 [=========================

> _**Note that you still haven't achieved convergence! From here, it's time to normalize the output data.**_

## Normalizing the output

Again, use `StandardScaler()` to: 

- Fit and transform `y_train` 
- Transform `y_val` and `y_test` 

In [10]:
# Instantiate StandardScaler
ss_y = StandardScaler()

# Fit and transform train labels
y_train_scaled = ss_y.fit_transform(y_train)

# Transform validate and test labels
y_val_scaled = ss_y.transform(y_val)
y_test_scaled = ss_y.transform(y_test)

In the cell below: 
- Train the `normalized_model` on normalized input (`X_train`) and output (`y_train_scaled`) 
- Set a batch size of 32 and train for 150 epochs 
- Specify the `validation_data` as `(X_val, y_val_scaled)` 

In [11]:
# Model with all normalized inputs and outputs
np.random.seed(123)
normalized_model = Sequential()
normalized_model.add(layers.Dense(100, activation='relu', input_shape=(n_features,)))
normalized_model.add(layers.Dense(50, activation='relu'))
normalized_model.add(layers.Dense(1, activation='linear'))

# Compile the model
normalized_model.compile(optimizer='SGD', 
                         loss='mse', 
                         metrics=['mse']) 

# Train the model
normalized_model.fit(X_train,y_train_scaled,batch_size=32,epochs=150,validation_data=(X_val,y_val_scaled))

Epoch 1/150
33/33 [==============================] - 0s 4ms/step - loss: 0.4587 - mse: 0.4587 - val_loss: 0.2263 - val_mse: 0.2263
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2628 - mse: 0.2628 - val_loss: 0.3224 - val_mse: 0.3224
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2196 - mse: 0.2196 - val_loss: 0.1586 - val_mse: 0.1586
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1871 - mse: 0.1871 - val_loss: 0.1363 - val_mse: 0.1363
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1618 - mse: 0.1618 - val_loss: 0.1305 - val_mse: 0.1305
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1508 - mse: 0.1508 - val_loss: 0.1290 - val_mse: 0.1290
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1354 - mse: 0.1354 - val_loss: 0.1263 - val_mse: 0.1263
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1275 - m

33/33 [==============================] - 0s 1ms/step - loss: 0.0248 - mse: 0.0248 - val_loss: 0.1277 - val_mse: 0.1277
Epoch 64/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0242 - mse: 0.0242 - val_loss: 0.1292 - val_mse: 0.1292
Epoch 65/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0235 - mse: 0.0235 - val_loss: 0.1305 - val_mse: 0.1305
Epoch 66/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0238 - mse: 0.0238 - val_loss: 0.1274 - val_mse: 0.1274
Epoch 67/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0230 - mse: 0.0230 - val_loss: 0.1271 - val_mse: 0.1271
Epoch 68/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0231 - mse: 0.0231 - val_loss: 0.1276 - val_mse: 0.1276
Epoch 69/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0223 - mse: 0.0223 - val_loss: 0.1273 - val_mse: 0.1273
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0220 - mse: 0

Epoch 125/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0120 - mse: 0.0120 - val_loss: 0.1363 - val_mse: 0.1363
Epoch 126/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0119 - mse: 0.0119 - val_loss: 0.1364 - val_mse: 0.1364
Epoch 127/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0115 - mse: 0.0115 - val_loss: 0.1355 - val_mse: 0.1355
Epoch 128/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0117 - mse: 0.0117 - val_loss: 0.1368 - val_mse: 0.1368
Epoch 129/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0114 - mse: 0.0114 - val_loss: 0.1377 - val_mse: 0.1377
Epoch 130/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0115 - mse: 0.0115 - val_loss: 0.1388 - val_mse: 0.1388
Epoch 131/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0112 - mse: 0.0112 - val_loss: 0.1398 - val_mse: 0.1398
Epoch 132/150
33/33 [==============================] - 0s 1ms/step - 

Nicely done! After normalizing both the input and output, the model finally converged. 

- Evaluate the model (`normalized_model`) on training data (`X_train` and `y_train_scaled`) 

In [12]:
# Evaluate the model on training data
normalized_model.evaluate(X_train,y_train_scaled)

33/33 [==============================] - 0s 516us/step - loss: 0.0087 - mse: 0.0087


[0.00874365121126175, 0.00874365121126175]

- Evaluate the model (`normalized_model`) on validate data (`X_val` and `y_val_scaled`) 

In [13]:
# Evaluate the model on validate data
normalized_model.evaluate(X_val,y_val_scaled)

9/9 [==============================] - 0s 550us/step - loss: 0.1382 - mse: 0.1382


[0.13824331760406494, 0.13824331760406494]

Since the output is normalized, the metric above is not interpretable. To remedy this: 

- Generate predictions on validate data (`X_val`) 
- Transform these predictions back to original scale using `ss_y` 
- Now you can calculate the RMSE in the original units with `y_val` and `y_val_pred` 

In [14]:
# Generate predictions on validate data
y_val_pred_scaled = normalized_model.predict(X_val)

# Transform the predictions back to original scale
y_val_pred = ss_y.inverse_transform(y_val_pred_scaled)

# RMSE of validate data
np.sqrt(mean_squared_error(y_val, y_val_pred))

29218.210488490113

Great! Now that you have a converged model, you can also experiment with alternative optimizers and initialization strategies to see if you can find a better global minimum. (After all, the current models may have converged to a local minimum.) 

## Using Weight Initializers

In this section you will to use alternative initialization and optimization strategies. At the end, you'll then be asked to select the model which you believe performs the best.  

##  He Initialization

In the cell below, sepcify the following in the first hidden layer:  
  - 100 units 
  - `'relu'` activation 
  - `input_shape` 
  - `kernel_initializer='he_normal'`  

In [16]:
np.random.seed(123)
he_model = Sequential()

# Add the first hidden layer
he_model.add(layers.Dense(100,activation='relu',kernel_initializer='he_normal',input_shape=(n_features,)))

# Add another hidden layer
he_model.add(layers.Dense(50, activation='relu'))

# Add an output layer
he_model.add(layers.Dense(1, activation='linear'))

# Compile the model
he_model.compile(optimizer='SGD', 
                 loss='mse', 
                 metrics=['mse'])

# Train the model
he_model.fit(X_train, 
             y_train_scaled, 
             batch_size=32, 
             epochs=150, 
             validation_data=(X_val, y_val_scaled))

Epoch 1/150
33/33 [==============================] - 0s 3ms/step - loss: 0.4973 - mse: 0.4973 - val_loss: 0.2717 - val_mse: 0.2717
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2550 - mse: 0.2550 - val_loss: 0.2092 - val_mse: 0.2092
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2143 - mse: 0.2143 - val_loss: 0.1834 - val_mse: 0.1834
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1795 - mse: 0.1795 - val_loss: 0.1686 - val_mse: 0.1686
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1626 - mse: 0.1626 - val_loss: 0.1636 - val_mse: 0.1636
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1449 - mse: 0.1449 - val_loss: 0.1571 - val_mse: 0.1571
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1301 - mse: 0.1301 - val_loss: 0.1614 - val_mse: 0.1614
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1264 - m

33/33 [==============================] - 0s 1ms/step - loss: 0.0271 - mse: 0.0271 - val_loss: 0.1588 - val_mse: 0.1588
Epoch 64/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0271 - mse: 0.0271 - val_loss: 0.1599 - val_mse: 0.1599
Epoch 65/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0267 - mse: 0.0267 - val_loss: 0.1589 - val_mse: 0.1589
Epoch 66/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0264 - mse: 0.0264 - val_loss: 0.1603 - val_mse: 0.1603
Epoch 67/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0257 - mse: 0.0257 - val_loss: 0.1591 - val_mse: 0.1591
Epoch 68/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0252 - mse: 0.0252 - val_loss: 0.1602 - val_mse: 0.1602
Epoch 69/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0249 - mse: 0.0249 - val_loss: 0.1610 - val_mse: 0.1610
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0250 - mse: 0

Epoch 125/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0129 - mse: 0.0129 - val_loss: 0.1723 - val_mse: 0.1723
Epoch 126/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0130 - mse: 0.0130 - val_loss: 0.1706 - val_mse: 0.1706
Epoch 127/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0128 - mse: 0.0128 - val_loss: 0.1702 - val_mse: 0.1702
Epoch 128/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0126 - mse: 0.0126 - val_loss: 0.1693 - val_mse: 0.1693
Epoch 129/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0123 - mse: 0.0123 - val_loss: 0.1699 - val_mse: 0.1699
Epoch 130/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0123 - mse: 0.0123 - val_loss: 0.1712 - val_mse: 0.1712
Epoch 131/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0123 - mse: 0.0123 - val_loss: 0.1710 - val_mse: 0.1710
Epoch 132/150
33/33 [==============================] - 0s 1ms/step - 

Evaluate the model (`he_model`) on training data (`X_train` and `y_train_scaled`) 

In [17]:
# Evaluate the model on training data
he_model.evaluate(X_train,y_train_scaled)

33/33 [==============================] - 0s 727us/step - loss: 0.0097 - mse: 0.0097


[0.009721646085381508, 0.009721646085381508]

Evaluate the model (`he_model`) on validate data (`X_train` and `y_train_scaled`) 

In [18]:
# Evaluate the model on validate data
he_model.evaluate(X_val,y_val_scaled)

9/9 [==============================] - 0s 545us/step - loss: 0.1730 - mse: 0.1730


[0.17303608357906342, 0.17303608357906342]

## Lecun Initialization 

In the cell below, sepcify the following in the first hidden layer:  
  - 100 units 
  - `'relu'` activation 
  - `input_shape` 
  - `kernel_initializer='lecun_normal'` 

In [19]:
np.random.seed(123)
lecun_model = Sequential()

# Add the first hidden layer
lecun_model.add(layers.Dense(100,activation='relu',kernel_initializer='lecun_normal',input_shape=(n_features,)))

# Add another hidden layer
lecun_model.add(layers.Dense(50, activation='relu'))

# Add an output layer
lecun_model.add(layers.Dense(1, activation='linear'))

# Compile the model
lecun_model.compile(optimizer='SGD', 
                    loss='mse', 
                    metrics=['mse'])

# Train the model
lecun_model.fit(X_train, 
                y_train_scaled, 
                batch_size=32, 
                epochs=150, 
                validation_data=(X_val, y_val_scaled))

Epoch 1/150
33/33 [==============================] - 0s 3ms/step - loss: 0.5014 - mse: 0.5014 - val_loss: 0.2456 - val_mse: 0.2456
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2315 - mse: 0.2315 - val_loss: 0.2210 - val_mse: 0.2210
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1870 - mse: 0.1870 - val_loss: 0.1576 - val_mse: 0.1576
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1621 - mse: 0.1621 - val_loss: 0.1362 - val_mse: 0.1362
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1509 - mse: 0.1509 - val_loss: 0.1313 - val_mse: 0.1313
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1330 - mse: 0.1330 - val_loss: 0.1196 - val_mse: 0.1196
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1204 - mse: 0.1204 - val_loss: 0.1143 - val_mse: 0.1143
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1173 - m

33/33 [==============================] - 0s 1ms/step - loss: 0.0254 - mse: 0.0254 - val_loss: 0.1221 - val_mse: 0.1221
Epoch 64/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0246 - mse: 0.0246 - val_loss: 0.1230 - val_mse: 0.1230
Epoch 65/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0242 - mse: 0.0242 - val_loss: 0.1281 - val_mse: 0.1281
Epoch 66/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0247 - mse: 0.0247 - val_loss: 0.1277 - val_mse: 0.1277
Epoch 67/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0237 - mse: 0.0237 - val_loss: 0.1254 - val_mse: 0.1254
Epoch 68/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0233 - mse: 0.0233 - val_loss: 0.1244 - val_mse: 0.1244
Epoch 69/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0232 - mse: 0.0232 - val_loss: 0.1258 - val_mse: 0.1258
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0229 - mse: 0

Epoch 125/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0126 - mse: 0.0126 - val_loss: 0.1368 - val_mse: 0.1368
Epoch 126/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0126 - mse: 0.0126 - val_loss: 0.1352 - val_mse: 0.1352
Epoch 127/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0127 - mse: 0.0127 - val_loss: 0.1346 - val_mse: 0.1346
Epoch 128/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0125 - mse: 0.0125 - val_loss: 0.1351 - val_mse: 0.1351
Epoch 129/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0122 - mse: 0.0122 - val_loss: 0.1346 - val_mse: 0.1346
Epoch 130/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0121 - mse: 0.0121 - val_loss: 0.1348 - val_mse: 0.1348
Epoch 131/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0120 - mse: 0.0120 - val_loss: 0.1357 - val_mse: 0.1357
Epoch 132/150
33/33 [==============================] - 0s 1ms/step - 

Evaluate the model (`lecun_model`) on training data (`X_train` and `y_train_scaled`) 

In [20]:
# Evaluate the model on training data
lecun_model.evaluate(X_train,y_train_scaled)

33/33 [==============================] - 0s 515us/step - loss: 0.0095 - mse: 0.0095


[0.009538842365145683, 0.009538842365145683]

Evaluate the model (`lecun_model`) on validate data (`X_train` and `y_train_scaled`) 

In [21]:
# Evaluate the model on validate data
lecun_model.evaluate(X_val,y_val_scaled)

9/9 [==============================] - 0s 631us/step - loss: 0.1362 - mse: 0.1362


[0.13617433607578278, 0.13617433607578278]

Not much of a difference, but a useful note to consider when tuning your network. Next, let's investigate the impact of various optimization algorithms.

## RMSprop 

Compile the `rmsprop_model` with: 

- `'rmsprop'` as the optimizer 
- track `'mse'` as the loss and metric  

In [22]:
np.random.seed(123)
rmsprop_model = Sequential()
rmsprop_model.add(layers.Dense(100, activation='relu', input_shape=(n_features,)))
rmsprop_model.add(layers.Dense(50, activation='relu'))
rmsprop_model.add(layers.Dense(1, activation='linear'))

# Compile the model
rmsprop_model.compile(optimizer='rmsprop',loss='mse',metrics=['mse'])

# Train the model
rmsprop_model.fit(X_train, 
                  y_train_scaled, 
                  batch_size=32, 
                  epochs=150, 
                  validation_data=(X_val, y_val_scaled))

Epoch 1/150
33/33 [==============================] - 0s 4ms/step - loss: 0.3563 - mse: 0.3563 - val_loss: 0.1590 - val_mse: 0.1590
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1707 - mse: 0.1707 - val_loss: 0.1314 - val_mse: 0.1314
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1265 - mse: 0.1265 - val_loss: 0.1160 - val_mse: 0.1160
Epoch 4/150
33/33 [==============================] - 0s 2ms/step - loss: 0.1231 - mse: 0.1231 - val_loss: 0.1116 - val_mse: 0.1116
Epoch 5/150
33/33 [==============================] - 0s 2ms/step - loss: 0.0870 - mse: 0.0870 - val_loss: 0.1263 - val_mse: 0.1263
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0778 - mse: 0.0778 - val_loss: 0.1153 - val_mse: 0.1153
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0626 - mse: 0.0626 - val_loss: 0.1219 - val_mse: 0.1219
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0448 - m

33/33 [==============================] - 0s 1ms/step - loss: 0.0089 - mse: 0.0089 - val_loss: 0.0941 - val_mse: 0.0941
Epoch 64/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0071 - mse: 0.0071 - val_loss: 0.0968 - val_mse: 0.0968
Epoch 65/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0075 - mse: 0.0075 - val_loss: 0.0916 - val_mse: 0.0916
Epoch 66/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0074 - mse: 0.0074 - val_loss: 0.0957 - val_mse: 0.0957
Epoch 67/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0102 - mse: 0.0102 - val_loss: 0.0889 - val_mse: 0.0889
Epoch 68/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0061 - mse: 0.0061 - val_loss: 0.1087 - val_mse: 0.1087
Epoch 69/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0060 - mse: 0.0060 - val_loss: 0.0880 - val_mse: 0.0880
Epoch 70/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0086 - mse: 0

Epoch 125/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0036 - mse: 0.0036 - val_loss: 0.0813 - val_mse: 0.0813
Epoch 126/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0052 - mse: 0.0052 - val_loss: 0.0913 - val_mse: 0.0913
Epoch 127/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0048 - mse: 0.0048 - val_loss: 0.0964 - val_mse: 0.0964
Epoch 128/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0067 - mse: 0.0067 - val_loss: 0.0799 - val_mse: 0.0799
Epoch 129/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0034 - mse: 0.0034 - val_loss: 0.0879 - val_mse: 0.0879
Epoch 130/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0063 - mse: 0.0063 - val_loss: 0.0801 - val_mse: 0.0801
Epoch 131/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0044 - mse: 0.0044 - val_loss: 0.0854 - val_mse: 0.0854
Epoch 132/150
33/33 [==============================] - 0s 1ms/step - 

Evaluate the model (`rmsprop_model`) on training data (`X_train` and `y_train_scaled`) 

In [23]:
# Evaluate the model on training data
rmsprop_model.evaluate(X_train,y_train_scaled)

33/33 [==============================] - 0s 491us/step - loss: 0.0028 - mse: 0.0028


[0.0028374691028147936, 0.0028374691028147936]

Evaluate the model (`rmsprop_model`) on training data (`X_train` and `y_train_scaled`) 

In [24]:
# Evaluate the model on validate data
rmsprop_model.evaluate(X_val,y_val_scaled)

9/9 [==============================] - 0s 550us/step - loss: 0.0777 - mse: 0.0777


[0.07766462117433548, 0.07766462117433548]

## Adam 

Compile the `adam_model` with: 

- `'Adam'` as the optimizer 
- track `'mse'` as the loss and metric  

In [25]:
np.random.seed(123)
adam_model = Sequential()
adam_model.add(layers.Dense(100, activation='relu', input_shape=(n_features,)))
adam_model.add(layers.Dense(50, activation='relu'))
adam_model.add(layers.Dense(1, activation='linear'))

# Compile the model
adam_model.compile(optimizer='Adam',loss='mse',metrics=['mse'])

# Train the model
adam_model.fit(X_train, 
               y_train_scaled, 
               batch_size=32, 
               epochs=150, 
               validation_data=(X_val, y_val_scaled))

Epoch 1/150
33/33 [==============================] - 0s 4ms/step - loss: 0.3527 - mse: 0.3527 - val_loss: 0.1585 - val_mse: 0.1585
Epoch 2/150
33/33 [==============================] - 0s 1ms/step - loss: 0.2159 - mse: 0.2159 - val_loss: 0.1081 - val_mse: 0.1081
Epoch 3/150
33/33 [==============================] - 0s 1ms/step - loss: 0.1094 - mse: 0.1094 - val_loss: 0.1106 - val_mse: 0.1106
Epoch 4/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0743 - mse: 0.0743 - val_loss: 0.1076 - val_mse: 0.1076
Epoch 5/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0589 - mse: 0.0589 - val_loss: 0.1003 - val_mse: 0.1003
Epoch 6/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0447 - mse: 0.0447 - val_loss: 0.1048 - val_mse: 0.1048
Epoch 7/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0366 - mse: 0.0366 - val_loss: 0.0945 - val_mse: 0.0945
Epoch 8/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0288 - m

33/33 [==============================] - 0s 2ms/step - loss: 0.0035 - mse: 0.0035 - val_loss: 0.0968 - val_mse: 0.0968
Epoch 64/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0039 - mse: 0.0039 - val_loss: 0.0995 - val_mse: 0.0995
Epoch 65/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0019 - mse: 0.0019 - val_loss: 0.0976 - val_mse: 0.0976
Epoch 66/150
33/33 [==============================] - 0s 2ms/step - loss: 0.0014 - mse: 0.0014 - val_loss: 0.0979 - val_mse: 0.0979
Epoch 67/150
33/33 [==============================] - 0s 1ms/step - loss: 9.0169e-04 - mse: 9.0169e-04 - val_loss: 0.0962 - val_mse: 0.0962
Epoch 68/150
33/33 [==============================] - 0s 1ms/step - loss: 6.1981e-04 - mse: 6.1981e-04 - val_loss: 0.0983 - val_mse: 0.0983
Epoch 69/150
33/33 [==============================] - 0s 1ms/step - loss: 5.5123e-04 - mse: 5.5123e-04 - val_loss: 0.0975 - val_mse: 0.0975
Epoch 70/150
33/33 [==============================] - 0s 1ms/step

Epoch 123/150
33/33 [==============================] - 0s 1ms/step - loss: 9.1329e-04 - mse: 9.1329e-04 - val_loss: 0.0954 - val_mse: 0.0954
Epoch 124/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0013 - mse: 0.0013 - val_loss: 0.0905 - val_mse: 0.0905
Epoch 125/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0019 - mse: 0.0019 - val_loss: 0.0980 - val_mse: 0.0980
Epoch 126/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0032 - mse: 0.0032 - val_loss: 0.0900 - val_mse: 0.0900
Epoch 127/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0052 - mse: 0.0052 - val_loss: 0.0987 - val_mse: 0.0987
Epoch 128/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0060 - mse: 0.0060 - val_loss: 0.0907 - val_mse: 0.0907
Epoch 129/150
33/33 [==============================] - 0s 1ms/step - loss: 0.0063 - mse: 0.0063 - val_loss: 0.0946 - val_mse: 0.0946
Epoch 130/150
33/33 [==============================] - 0s 1ms

Evaluate the model (`adam_model`) on training data (`X_train` and `y_train_scaled`) 

In [26]:
# Evaluate the model on training data
adam_model.evaluate(X_train,y_train_scaled)

33/33 [==============================] - 0s 500us/step - loss: 0.0032 - mse: 0.0032


[0.00323811499401927, 0.00323811499401927]

Evaluate the model (`adam_model`) on training data (`X_train` and `y_train_scaled`) 

In [27]:
# Evaluate the model on validate data
adam_model.evaluate(X_val,y_val_scaled)

9/9 [==============================] - 0s 539us/step - loss: 0.0953 - mse: 0.0953


[0.09525203704833984, 0.09525203704833984]

## Select a Final Model

Now, select the model with the best performance based on the training and validation sets. Evaluate this top model using the test set!

In [28]:
# Evaluate the best model on test data
rmsprop_model.evaluate(X_test,y_test_scaled)

5/5 [==============================] - 0s 622us/step - loss: 0.1762 - mse: 0.1762


[0.1762213557958603, 0.1762213557958603]

As earlier, this metric is hard to interpret because the output is scaled. 

- Generate predictions on test data (`X_test`) 
- Transform these predictions back to original scale using `ss_y` 
- Now you can calculate the RMSE in the original units with `y_test` and `y_test_pred` 

In [29]:
# Generate predictions on test data
y_test_pred_scaled = rmsprop_model.predict(X_test)

# Transform the predictions back to original scale
y_test_pred = ss_y.inverse_transform(y_test_pred_scaled)

# MSE of test data
np.sqrt(mean_squared_error(y_test, y_test_pred))


32988.36159306616

## Summary  

In this lab, you worked to ensure your model converged properly by normalizing both the input and output. Additionally, you also investigated the impact of varying initialization and optimization routines.